In [1]:
!pip install pymongo pandas
import pandas as pd
from pymongo import MongoClient
from bson import ObjectId

MONGO_URI = "mongodb://10.3.32.15:27017/"
MONGO_DB = "main"
COLLECTION_NAME = "tracking_sessions_v2"

client = MongoClient(MONGO_URI)
db = client[MONGO_DB]
collection = db[COLLECTION_NAME]

print(f"Connected to {MONGO_DB}.{COLLECTION_NAME}")


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Connected to main.tracking_sessions_v2


In [2]:
# Read buildingIDs from CSV
df = pd.read_csv('addresses_with_buildingIDs.csv')
print(f"Loaded {len(df)} addresses with buildingIDs")
print(df.head())

# Get unique buildingIDs that were found (exclude 'NOT_FOUND')
building_ids = df[df['BuildingID'] != 'NOT_FOUND']['BuildingID'].unique()
print(f"\nQuerying for {len(building_ids)} unique buildingIDs")

Loaded 640 addresses with buildingIDs
                              Address                BuildingID
0    713 E 211th St, Bronx, NY, 10467  69351bbd5a54c51c3d607e34
1   1609 E 174th St, Bronx, NY, 10472  69351b9e349b1fe3f3a5b8ee
2  1330 Webster Ave, Bronx, NY, 10456  69351c849c44cab1c93c9c7e
3      4439 3rd Ave, Bronx, NY, 10457  69351bc19da13d9bfeea5d96
4    530 E 169th St, Bronx, NY, 10456  69351bcbdd0e0ee2f5c5b6ab

Querying for 578 unique buildingIDs


In [3]:
# Iterate through buildingIDs and find matching UUIDs
results = []

for building_id in building_ids:
    # Use buildingID as string, not ObjectId. Project only uuid field
    docs = collection.find(
        {
            '$or': [
                {'buildingData.targetBuilding.buildingID': building_id},
                {'buildingData.detectedBuilding.buildingID': building_id}
            ]
        },
        {'uuid': 1}  # Project only uuid field
    )
    
    for doc in docs:
        uuid = doc.get('uuid', 'NOT_FOUND')
        results.append({'BuildingID': building_id, 'UUID': uuid})
        print(f"Found: {building_id} -> {uuid}")

results_df = pd.DataFrame(results)
print(f"\nTotal sessions found: {len(results_df)}")
print(results_df.head(10))

Found: 69351bbd5a54c51c3d607e34 -> 097d333e-44ff-4eca-b243-eb8e2e9ea453
Found: 69351bbd5a54c51c3d607e34 -> 217048d5-97ce-48f3-b6da-02c7c92b1d64
Found: 69351bbd5a54c51c3d607e34 -> 2a57d46a-c552-4b6e-ba09-31e5f44b0279
Found: 69351bbd5a54c51c3d607e34 -> 396c1952-d233-46c6-80d8-ad67b4e263d8
Found: 69351bbd5a54c51c3d607e34 -> 3b469195-68a1-4cf9-8b27-e1bc3252311b
Found: 69351bbd5a54c51c3d607e34 -> 3ec19e61-8ce4-487f-b70d-b4ff7dd24191
Found: 69351bbd5a54c51c3d607e34 -> 7f020cf7-f03e-46f5-a927-02fb0210fb4f
Found: 69351bbd5a54c51c3d607e34 -> a6a4b3c2-e315-46c9-8d21-74c8d683fc59
Found: 69351bbd5a54c51c3d607e34 -> ae3902a2-8a25-4ba1-baa9-0d03cfbb6769
Found: 69351bbd5a54c51c3d607e34 -> e16a2cc8-15a4-4e40-83fa-22f4306a5cfb
Found: 69351bbd5a54c51c3d607e34 -> e7bea9af-2845-4c99-ae89-3153918e91c3
Found: 69351bbd5a54c51c3d607e34 -> f45ac8b2-d56e-4511-9714-1dcf49f8c270
Found: 69351bbd5a54c51c3d607e34 -> f8e7e12b-e5fe-4b2d-80f5-e445778f10f6
Found: 69351bbd5a54c51c3d607e34 -> cc4c910b-20fe-4b5b-8072-daf82

In [4]:
# Save results to CSV
output_path = 'buildingID_to_uuid.csv'
results_df.to_csv(output_path, index=False)
print(f"Results saved to {output_path}")
print(f"\nTotal buildingIDs processed: {len(building_ids)}")
print(f"Sessions found: {len(results_df)}")
print(f"\nUnique buildingIDs with sessions: {results_df['BuildingID'].nunique()}")

Results saved to buildingID_to_uuid.csv

Total buildingIDs processed: 578
Sessions found: 24533

Unique buildingIDs with sessions: 553
